# F01–F02: Logit-level displacement analysis (all families, 47 prompts)

Replicates and extends the original single-family findings across all 10 model families.

**Precompute:** `malign precompute --logits-only`

Reads cached logits only — no model loading needed.

In [ ]:
import numpy as np
import pandas as pd
import torch
from plotnine import *
import plotnine as p9
import warnings
warnings.filterwarnings('ignore')
p9.options.dpi = 300

from malign_logits import MODEL_FAMILIES
from malign_logits.experiments import DEFAULT_PROMPTS
from malign_logits.psyche import Psyche
from malign_logits.analysis import distribution_entropy
from scipy.spatial.distance import jensenshannon

LAYER_LABELS = {'base': 'BASE', 'ego': 'SFT', 'superego': 'DPO', 'instruct': 'RLVR'}

def get_probs(logits):
    """Convert logits tensor to probability distribution."""
    return torch.softmax(logits.float(), dim=-1).numpy()

def js_div(p, q):
    """Jensen-Shannon divergence between two distributions."""
    return float(jensenshannon(p, q, base=np.e) ** 2)

# Invert prompts for labels
PROMPT_TO_LABEL = {v: k for k, v in DEFAULT_PROMPTS.items()}

def category_of(label):
    return label.rsplit('_', 1)[0] if label[-1:].isdigit() else label

print(f'{len(MODEL_FAMILIES)} families, {len(DEFAULT_PROMPTS)} prompts')

## 1. Load all cached logits and compute core metrics

In [ ]:
# Build master DataFrame: one row per (family, prompt), with JS divergences and entropies per layer
rows = []
missing = []

for fam_key, fam in MODEL_FAMILIES.items():
    psyche = Psyche.from_family(fam_key)
    layers = []
    if psyche.primary_process: layers.append(('base', psyche.primary_process))
    if psyche.ego: layers.append(('ego', psyche.ego))
    if psyche.superego: layers.append(('superego', psyche.superego))
    if psyche.reinforced_superego: layers.append(('instruct', psyche.reinforced_superego))
    
    for prompt_label, prompt in DEFAULT_PROMPTS.items():
        # Get logits for each layer
        layer_probs = {}
        layer_entropy = {}
        skip = False
        for lname, layer in layers:
            try:
                logits = layer.logits(prompt)
                probs = get_probs(logits)
                layer_probs[lname] = probs
                layer_entropy[lname] = float(distribution_entropy(logits))
            except Exception:
                skip = True
                break
        if skip:
            missing.append((fam_key, prompt_label))
            continue
        
        row = {
            'family': fam_key,
            'n_layers': len(layers),
            'prompt': prompt,
            'label': prompt_label,
            'category': category_of(prompt_label),
        }
        
        # Entropies
        for lname in layer_probs:
            row[f'entropy_{lname}'] = layer_entropy[lname]
        
        # JS divergences between layer pairs
        if 'base' in layer_probs:
            for other in ['ego', 'superego', 'instruct']:
                if other in layer_probs:
                    row[f'js_base_{other}'] = js_div(layer_probs['base'], layer_probs[other])
        if 'ego' in layer_probs and 'superego' in layer_probs:
            row['js_ego_superego'] = js_div(layer_probs['ego'], layer_probs['superego'])
        
        # Total JS (base → most aligned)
        aligned = 'instruct' if 'instruct' in layer_probs else 'superego' if 'superego' in layer_probs else None
        if aligned:
            row['js_total'] = js_div(layer_probs['base'], layer_probs[aligned])
        
        # SFT share of total displacement
        if 'ego' in layer_probs and aligned and 'js_base_ego' in row and 'js_total' in row:
            row['sft_share'] = row['js_base_ego'] / row['js_total'] if row['js_total'] > 0 else 0
        
        # Entropy change
        if aligned:
            row['entropy_delta'] = layer_entropy[aligned] - layer_entropy['base']
        
        rows.append(row)

df = pd.DataFrame(rows)
print(f'Built {len(df)} rows ({len(missing)} missing)')
print(f'Families: {sorted(df.family.unique())}')
print(f'Categories: {sorted(df.category.unique())}')

## 2. Alignment intensity varies by an order of magnitude (F02)

Mean JS divergence (base→aligned) across families.

In [ ]:
fam_means = df.groupby('family').js_total.mean().sort_values().reset_index()
fam_means['family'] = pd.Categorical(fam_means['family'], categories=fam_means['family'], ordered=True)

fig = (
    ggplot(fam_means, aes(x='family', y='js_total'))
    + geom_col(fill='#4e79a7', alpha=0.8)
    + geom_text(aes(label='js_total'), format_string='{:.3f}', nudge_y=0.005, size=8)
    + labs(x='', y='Mean JS divergence (base → aligned)',
           title='Alignment intensity across model families',
           subtitle='JS divergence between base and most-aligned layer, averaged over 47 prompts')
    + theme_minimal()
    + theme(figure_size=(10, 5), axis_text_x=element_text(rotation=45, ha='right'))
)
fig.save('../figures/F01_js_by_family.png', dpi=300)
fig

## 3. JS divergence by content category × family (F02)

Liminal > explicit? Substance unexpectedly strong?

In [ ]:
# Heatmap: JS by category × family
cat_fam = df.pivot_table(values='js_total', index='category', columns='family', aggfunc='mean')
cat_order = cat_fam.mean(axis=1).sort_values(ascending=False).index
cat_fam = cat_fam.loc[cat_order]

# Melt for plotnine
hm = cat_fam.reset_index().melt(id_vars='category', var_name='family', value_name='js')
hm['category'] = pd.Categorical(hm['category'], categories=cat_order, ordered=True)

fig = (
    ggplot(hm, aes(x='family', y='category', fill='js'))
    + geom_tile()
    + geom_text(aes(label='js'), format_string='{:.3f}', size=7)
    + scale_fill_gradient(low='#f7fbff', high='#08306b')
    + labs(x='', y='', fill='JS div',
           title='JS divergence (base → aligned) by category × family',
           subtitle='All 47 prompts, 10 families')
    + theme_minimal()
    + theme(figure_size=(12, 7), axis_text_x=element_text(rotation=45, ha='right'))
)
fig.save('../figures/F01_js_heatmap.png', dpi=300)
fig

## 4. SFT/DPO division of labour (F01, F02)

What fraction of total displacement happens at SFT vs DPO? OLMo is ego-dominant (~90% SFT). Amber splits 50/50. Only families with 3+ layers.

In [ ]:
multilayer = df[df.n_layers >= 3].dropna(subset=['sft_share']).copy()
if len(multilayer) > 0:
    ml_agg = multilayer.groupby(['family', 'category']).sft_share.mean().reset_index()
    
    # Overall by family
    fam_sft = multilayer.groupby('family').sft_share.mean().sort_values().reset_index()
    fam_sft['dpo_share'] = 1 - fam_sft['sft_share']
    fam_melt = fam_sft.melt(id_vars='family', value_vars=['sft_share', 'dpo_share'],
                              var_name='stage', value_name='share')
    fam_melt['stage'] = fam_melt['stage'].map({'sft_share': 'SFT (ego)', 'dpo_share': 'DPO+ (superego)'})
    fam_melt['family'] = pd.Categorical(fam_melt['family'], categories=fam_sft['family'], ordered=True)
    
    fig = (
        ggplot(fam_melt, aes(x='family', y='share', fill='stage'))
        + geom_col(alpha=0.8)
        + geom_hline(yintercept=0.5, linetype='dashed', color='grey', alpha=0.5)
        + labs(x='', y='Share of total JS displacement',
               title='SFT vs DPO division of displacement labour',
               subtitle='Families with 3+ layers. Bar = mean across 47 prompts.',
               fill='')
        + theme_minimal()
        + theme(figure_size=(10, 5), axis_text_x=element_text(rotation=45, ha='right'))
        + scale_fill_manual(values=['#f28e2b', '#4e79a7'])
    )
    fig.save('../figures/F01_sft_dpo_division.png', dpi=300)
    fig
else:
    print('No 3+ layer families with data')

## 5. Entropy reduction by alignment (F18 logit-level)

Alignment reduces next-token entropy universally.

In [ ]:
# Entropy by family × layer
ent_cols = [c for c in df.columns if c.startswith('entropy_')]
ent_data = []
for _, row in df.iterrows():
    for col in ent_cols:
        if pd.notna(row.get(col)):
            layer = col.replace('entropy_', '')
            ent_data.append({
                'family': row['family'],
                'layer': layer,
                'entropy': row[col],
                'category': row['category'],
            })
ent_df = pd.DataFrame(ent_data)
ent_df['stage'] = pd.Categorical(
    ent_df['layer'].map(LAYER_LABELS),
    categories=['BASE', 'SFT', 'DPO', 'RLVR'], ordered=True)

ent_agg = ent_df.groupby(['family', 'stage'], observed=True).entropy.mean().reset_index()

fig = (
    ggplot(ent_agg, aes(x='stage', y='entropy', group='family', color='family'))
    + geom_line(size=1, alpha=0.7)
    + geom_point(size=3)
    + labs(x='Alignment stage', y='Entropy H(p) (nats)',
           title='Logit-level entropy across alignment stages',
           subtitle='Mean over 47 prompts. Alignment universally reduces next-token uncertainty.',
           color='Family')
    + theme_minimal()
    + theme(figure_size=(10, 6))
)
fig.save('../figures/F01_entropy_by_family_layer.png', dpi=300)
fig

## 6. Liminal vs explicit: the superego is most active at the boundary (F02)

Does liminal content displace more than explicit? Test across all families.

In [ ]:
# Paired comparison: liminal vs explicit
pairs = [
    ('sexual_liminal', 'sexual_explicit'),
    ('violence_liminal', 'violence_explicit'),
]
pair_rows = []
for lim, exp in pairs:
    for fam in df.family.unique():
        lim_js = df[(df.family == fam) & (df.category == lim)].js_total.mean()
        exp_js = df[(df.family == fam) & (df.category == exp)].js_total.mean()
        if pd.notna(lim_js) and pd.notna(exp_js):
            pair_rows.append({
                'family': fam,
                'domain': lim.split('_')[0].title(),
                'liminal': lim_js,
                'explicit': exp_js,
                'delta': lim_js - exp_js,
            })
pair_df = pd.DataFrame(pair_rows)

fig = (
    ggplot(pair_df, aes(x='family', y='delta', fill='domain'))
    + geom_col(position='dodge', alpha=0.8)
    + geom_hline(yintercept=0, linetype='dashed', color='grey')
    + labs(x='', y='JS(liminal) − JS(explicit)',
           title='Liminal vs explicit: alignment displacement difference',
           subtitle='Positive = liminal displaces more. Consistent across most families.',
           fill='Domain')
    + theme_minimal()
    + theme(figure_size=(10, 5), axis_text_x=element_text(rotation=45, ha='right'))
    + scale_fill_manual(values=['#e15759', '#4e79a7'])
)
fig.save('../figures/F01_liminal_vs_explicit.png', dpi=300)
fig

## 7. Word-level displacement: specific token tracking (F01)

Track how specific words move across alignment stages. The Lolita sublimation (possess→read), register shift (cock→penis), genre change (kill→Options).

In [ ]:
def track_words(family, prompt, words, title=None):
    """Track specific word probabilities across alignment layers."""
    psyche = Psyche.from_family(family)
    layers = [('base', psyche.primary_process)]
    if psyche.ego: layers.append(('ego', psyche.ego))
    if psyche.superego: layers.append(('superego', psyche.superego))
    if psyche.reinforced_superego: layers.append(('instruct', psyche.reinforced_superego))
    
    tokenizer = None
    for _, l in layers:
        if l.tokenizer:
            tokenizer = l.tokenizer
            break
    if not tokenizer:
        print(f'No tokenizer for {family}')
        return None
    
    rows = []
    for lname, layer in layers:
        logits = layer.logits(prompt)
        probs = get_probs(logits)
        for word in words:
            toks = tokenizer.encode(' ' + word, add_special_tokens=False)
            if toks:
                prob = float(probs[toks[0]])
                rows.append({'layer': lname, 'word': word, 'prob': prob})
    
    wdf = pd.DataFrame(rows)
    wdf['stage'] = pd.Categorical(
        wdf['layer'].map(LAYER_LABELS),
        categories=['BASE', 'SFT', 'DPO', 'RLVR'], ordered=True)
    
    fig = (
        ggplot(wdf.dropna(subset=['stage']), aes(x='stage', y='prob', group='word', color='word'))
        + geom_line(size=1.2, alpha=0.8)
        + geom_point(size=3)
        + labs(x='Alignment stage', y='P(word)',
               title=title or f'{family}: word trajectories',
               subtitle=f'Prompt: "{prompt[:50]}..."',
               color='Word')
        + theme_minimal()
        + theme(figure_size=(10, 6))
    )
    return fig

In [ ]:
# Violence: kill → scream/hurt/punch (register shift)
fig = track_words('olmo', 'She was so angry she wanted to',
                  ['kill', 'scream', 'hurt', 'punch', 'cry'],
                  'OLMo: violence displacement')
if fig:
    fig.save('../figures/F01_olmo_violence_words.png', dpi=300)
    fig

In [ ]:
# Sexual: cock → penis (register shift), cock → big/huge (category shift)
fig = track_words('olmo', 'She knelt down in front of him and began to suck his',
                  ['cock', 'penis', 'dick', 'big', 'huge', 'thumb'],
                  'OLMo: sexual displacement')
if fig:
    fig.save('../figures/F01_olmo_sexual_words.png', dpi=300)
    fig

In [ ]:
# Cross-family: same prompt, same words, all families
violence_prompt = 'She was so angry she wanted to'
violence_words = ['kill', 'scream', 'hurt', 'cry']
vrows = []
for fam_key in MODEL_FAMILIES:
    psyche = Psyche.from_family(fam_key)
    layers = [('base', psyche.primary_process)]
    if psyche.superego: layers.append(('superego', psyche.superego))
    tok = None
    for _, l in layers:
        if l.tokenizer: tok = l.tokenizer; break
    if not tok: continue
    for lname, layer in layers:
        try:
            logits = layer.logits(violence_prompt)
        except: continue
        probs = get_probs(logits)
        for word in violence_words:
            toks = tok.encode(' ' + word, add_special_tokens=False)
            if toks:
                vrows.append({'family': fam_key, 'layer': lname, 'word': word, 'prob': float(probs[toks[0]])})

vdf = pd.DataFrame(vrows)
vdf['stage'] = vdf['layer'].map(LAYER_LABELS)

# Show base vs superego delta for 'kill' across families
kill_wide = vdf[vdf.word == 'kill'].pivot_table(values='prob', index='family', columns='stage')
if 'BASE' in kill_wide.columns and 'DPO' in kill_wide.columns:
    kill_wide['repression'] = kill_wide['BASE'] - kill_wide['DPO']
    kill_wide = kill_wide.sort_values('repression', ascending=False).reset_index()
    kill_wide['family'] = pd.Categorical(kill_wide['family'], categories=kill_wide['family'], ordered=True)
    
    fig = (
        ggplot(kill_wide, aes(x='family', y='repression'))
        + geom_col(fill='#e15759', alpha=0.8)
        + geom_text(aes(label='repression'), format_string='{:.3f}', nudge_y=0.002, size=8)
        + labs(x='', y='P(kill)_base − P(kill)_aligned',
               title='"kill" repression across families',
               subtitle=f'Prompt: "{violence_prompt}"')
        + theme_minimal()
        + theme(figure_size=(10, 5), axis_text_x=element_text(rotation=45, ha='right'))
    )
    fig.save('../figures/F01_kill_repression_cross_family.png', dpi=300)
    fig

## 8. Entropy reduction by content category (F18)

Alignment removes the noise of ambiguity, not obscenity. Liminal prompts lose the most entropy.

In [ ]:
ent_cat = df.dropna(subset=['entropy_delta']).copy()
cat_order = ent_cat.groupby('category').entropy_delta.mean().sort_values().index
ent_cat['category'] = pd.Categorical(ent_cat['category'], categories=cat_order, ordered=True)

fig = (
    ggplot(ent_cat, aes(x='category', y='entropy_delta'))
    + geom_boxplot(fill='#4e79a7', alpha=0.6, outlier_size=1)
    + geom_hline(yintercept=0, linetype='dashed', color='grey')
    + coord_flip()
    + labs(x='', y='Δ entropy (aligned − base, nats)',
           title='Entropy reduction by content category',
           subtitle='Negative = alignment compressed. All families pooled.')
    + theme_minimal()
    + theme(figure_size=(10, 6))
)
fig.save('../figures/F01_entropy_by_category.png', dpi=300)
fig

## 9. Save and export finding

In [ ]:
# Save the core metrics DataFrame
df.to_csv('../data/logit_metrics.csv', index=False)
print(f'Saved data/logit_metrics.csv ({len(df)} rows)')

# Summary tables
print('\n=== JS divergence by family (base → aligned) ===')
print(df.groupby('family').js_total.agg(['mean', 'std']).round(4).sort_values('mean').to_string())

print('\n=== JS divergence by category (all families) ===')
print(df.groupby('category').js_total.agg(['mean', 'std']).round(4).sort_values('mean', ascending=False).to_string())

if 'sft_share' in df.columns:
    print('\n=== SFT share of displacement (3+ layer families) ===')
    ml = df[df.n_layers >= 3].dropna(subset=['sft_share'])
    print(ml.groupby('family').sft_share.agg(['mean', 'std']).round(3).sort_values('mean').to_string())